In [ ]:
# ¿Cómo se organizan ventas y utilidad por fecha, cliente, producto y geografía sin perder el grano de línea de orden?

import sqlite3
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
DATA = ROOT / "data"
MART = ROOT / "submission" / "superstore_mart.db"

In [ ]:
# El hecho conserva una línea de orden; las dimensiones permiten filtrar y agregar sin mezclar granos.

orders = pd.read_csv(DATA / "orders.csv")
customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
order_lines = pd.read_csv(DATA / "order_lines.csv")

In [ ]:
# Las fechas se derivan de la orden para que el mart responda comparaciones temporales.

dim_date = orders[["Order Date"]].drop_duplicates().reset_index(drop=True)
dim_date["date_key"] = dim_date.index + 1
dim_date["date"] = pd.to_datetime(dim_date["Order Date"], format="%d/%m/%y")
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
orders = orders.merge(dim_date[["Order Date", "date_key"]], on="Order Date", validate="many_to_one")

In [ ]:
# La tabla de hechos referencia las dimensiones y mantiene las medidas en el nivel de una línea de orden.

fact_sales = order_lines.merge(orders[["order_context_key", "date_key"]], on="order_context_key", validate="many_to_one")
assert len(fact_sales) == len(order_lines)

with sqlite3.connect(MART) as database:
    customers.to_sql("dim_customer", database, index=False, if_exists="replace")
    products.to_sql("dim_product", database, index=False, if_exists="replace")
    dim_date[["date_key", "Order Date", "year", "month"]].to_sql("dim_date", database, index=False, if_exists="replace")
    fact_sales.to_sql("fact_sales", database, index=False, if_exists="replace")

    monthly = pd.read_sql_query("""
        SELECT d.year, d.month, p.`Product Category`, SUM(f.Sales) AS sales, SUM(f.Profit) AS profit
        FROM fact_sales f
        JOIN dim_date d USING(date_key)
        JOIN dim_product p USING(product_context_key)
        GROUP BY d.year, d.month, p.`Product Category`
        ORDER BY d.year, d.month, sales DESC
    """, database)
monthly.head()